In [166]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import GammaRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error

In [167]:
# Load cleaned data
df = pd.read_csv('./Cleaned Data/cleaned_data.csv')

In [168]:
# Keep only positive targets to avoid issues with log transformation so the model won't encounter zero or negative values
df = df[df['Total Time Spent'] > 0].reset_index(drop=True)

In [169]:
# define features and target variables
text_features = 'Summary'
categorical_features = ['Issue Type', 'Priority']
target = 'Total Time Spent'

In [170]:
X = df[[text_features] + categorical_features]
y = df[target]

In [171]:
# log-transform the target variable to prevent the model from predicting negative values
y_log = np.log1p(y)

In [172]:
# prepare feature matrix X and target vector y
final_preprocessor = ColumnTransformer(
    transformers=[
        (
            'text',
            TfidfVectorizer(
                max_features=300,
                min_df=3,
                ngram_range=(1, 2)
            ),
            'Summary'
        ),
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_features
        )
    ]
)

In [173]:
from sklearn.model_selection import GridSearchCV

# use hyperparameter tuning to find the best parameters for the model

param_grid = {
    'regressor__alpha': np.logspace(-4, 2, 20),
    'regressor__max_iter': [1000, 2000, 3000]
}
model_for_tuning = Pipeline(
    steps=[
        ('preprocessor', final_preprocessor),
        ('regressor', GammaRegressor())
    ]
)
grid_search = GridSearchCV(
    model_for_tuning,
    param_grid,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

grid_search.fit(X, y_log)
print("Best parameters found: ", grid_search.best_params_['regressor__alpha'], grid_search.best_params_['regressor__max_iter'])
print("Best CV MAE: ", -grid_search.best_score_)

Best parameters found:  0.0001 1000
Best CV MAE:  0.6627949454699383


In [174]:
# create the final model pipeline
model = Pipeline(
    steps=[
        ('preprocessor', final_preprocessor),
        ('regressor', GammaRegressor(
            alpha=grid_search.best_params_['regressor__alpha'],
            max_iter=grid_search.best_params_['regressor__max_iter']
        ))
    ]
)


In [175]:
# use K-Fold cross-validation to evaluate the model
# this provides a more robust estimate of model performance
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    model,
    X,
    y_log,
    cv=kf,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

mae_scores = -cv_scores

print("K-Fold MAE scores:", mae_scores)
print(f"Average MAE: {mae_scores.mean():.2f}")
print(f"Std Dev: {mae_scores.std():.2f}")

K-Fold MAE scores: [0.66956724 0.68329858 0.67322059 0.6755314  0.69841089]
Average MAE: 0.68
Std Dev: 0.01


In [176]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

In [177]:
# Log transform target
y_train_log = np.log1p(y_train)

In [178]:
# Fit model
model.fit(X_train, y_train_log)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('text', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [179]:
# Predict
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)

In [180]:
results = pd.DataFrame({
    'Summary': df.loc[X_test.index, 'Summary'],
    'Issue Type': df.loc[X_test.index, 'Issue Type'],
    'Priority': df.loc[X_test.index, 'Priority'],
    'Original Estimated Time': df.loc[X_test.index, 'Original Time Estimated'],
    'Actual Time Spent': y_test,
    'Predicted Time Spent': y_pred
})

In [181]:
results

,Summary,Issue Type,Priority,Original Estimated Time,Actual Time Spent,Predicted Time Spent
5601,"['Add', 'endpoint', 'upserting', 'disciplines']",Subtask,Major,60,60,101.375942
2108,"['Add', 'endpoint', 'saving', 'configuration']",Subtask,Major,120,205,187.739912
2516,"['I', 'get', 'analysis', 'AR', 'I', 'first', '...",Story,Major,0,75,63.686130
7318,['Feedback'],Subtask,Major,0,20,81.134466
33,"['What', 'caused', 'change', 'value', 'invento...",Subtask,Major,0,141,41.929809
...,...,...,...,...,...,...
4269,"['As', 'child', 'I', 'able', 'login', 'code']",Story,Major,0,120,164.918146
6762,"['update', 'PO', 'method', 'approve', 'POs']",Subtask,Major,30,60,74.465252
6965,"['Upgrade', 'version', 'opentelemetry', 'use']",Task,Critical,0,240,77.770085
4184,"['add', 'ela', 'writing', 'subject']",Subtask,Major,120,300,75.031024


In [182]:
results['Absolute Error Predicted'] = np.abs(
    results['Actual Time Spent'] - results['Predicted Time Spent']
)


In [183]:
# absolute error original
results['Absolute Error Original'] = np.abs(
    results['Actual Time Spent'] - results['Original Estimated Time']
)

In [184]:
results

,Summary,Issue Type,Priority,Original Estimated Time,Actual Time Spent,Predicted Time Spent,Absolute Error Predicted,Absolute Error Original
5601,"['Add', 'endpoint', 'upserting', 'disciplines']",Subtask,Major,60,60,101.375942,41.375942,0
2108,"['Add', 'endpoint', 'saving', 'configuration']",Subtask,Major,120,205,187.739912,17.260088,85
2516,"['I', 'get', 'analysis', 'AR', 'I', 'first', '...",Story,Major,0,75,63.686130,11.313870,75
7318,['Feedback'],Subtask,Major,0,20,81.134466,61.134466,20
33,"['What', 'caused', 'change', 'value', 'invento...",Subtask,Major,0,141,41.929809,99.070191,141
...,...,...,...,...,...,...,...,...
4269,"['As', 'child', 'I', 'able', 'login', 'code']",Story,Major,0,120,164.918146,44.918146,120
6762,"['update', 'PO', 'method', 'approve', 'POs']",Subtask,Major,30,60,74.465252,14.465252,30
6965,"['Upgrade', 'version', 'opentelemetry', 'use']",Task,Critical,0,240,77.770085,162.229915,240
4184,"['add', 'ela', 'writing', 'subject']",Subtask,Major,120,300,75.031024,224.968976,180


In [185]:
# mean absolute error original
mae_original = results['Absolute Error Original'].mean()
print(f"Mean Absolute Error (Original): {mae_original:.2f}")

Mean Absolute Error (Original): 90.41


In [186]:
# mean absolute error predicted
mae_predicted = results['Absolute Error Predicted'].mean()
print(f"Mean Absolute Error (Predicted): {mae_predicted:.2f}")

Mean Absolute Error (Predicted): 78.95


In [187]:
results.to_csv('prediction_results.csv', index=False)